# 01 — Data Ingestion

This notebook loads data from three sources:
1. **Toronto Bikeshare ridership** — monthly CSVs (Jan–Oct 2025), already downloaded
2. **Station metadata** — from Toronto Open Data CKAN API (capacity, lat/lon)
3. **Historical weather** — from Open-Meteo API (temperature, precipitation, wind speed)

All raw data is saved to `../data/raw/` for downstream cleaning.

In [3]:
import pandas as pd
import numpy as np
import requests
import os
from pathlib import Path

# Paths
PROJECT_DIR = Path("..").resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"
BIKESHARE_SRC = PROJECT_DIR / "bikeshare-ridership-2025"

RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project dir: {PROJECT_DIR}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Source CSVs: {BIKESHARE_SRC}")

Project dir: C:\Users\abdul\Documents\ECE 1513
Raw data dir: C:\Users\abdul\Documents\ECE 1513\data\raw
Source CSVs: C:\Users\abdul\Documents\ECE 1513\bikeshare-ridership-2025


## 1. Load Bikeshare Ridership Data
Load all monthly CSVs (Jan–Oct 2025), concatenate into a single dataframe, and do a quick sanity check.

In [6]:
# Load all monthly CSVs
csv_files = sorted(BIKESHARE_SRC.glob("bikeshare_2025_*.csv"))
print(f"Found {len(csv_files)} CSV files")

dfs = []
for f in csv_files:
    df = pd.read_csv(f, parse_dates=["Start_Time", "End_Time"], encoding="latin-1")
    month = f.stem.split("_")[-1]
    print(f"  {f.name}: {len(df):,} rows")
    dfs.append(df)

rides = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rides: {len(rides):,}")
print(f"Date range: {rides['Start_Time'].min()} → {rides['Start_Time'].max()}")
rides.head()

Found 10 CSV files
  bikeshare_2025_01.csv: 202,946 rows
  bikeshare_2025_02.csv: 125,670 rows
  bikeshare_2025_03.csv: 312,178 rows
  bikeshare_2025_04.csv: 471,525 rows
  bikeshare_2025_05.csv: 699,461 rows
  bikeshare_2025_06.csv: 980,624 rows
  bikeshare_2025_07.csv: 1,192,062 rows
  bikeshare_2025_08.csv: 1,164,697 rows
  bikeshare_2025_09.csv: 1,062,001 rows
  bikeshare_2025_10.csv: 872,482 rows

Total rides: 7,083,646
Date range: 2025-01-01 00:01:51 → 2025-10-31 23:59:59


,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,34637659,193,7192.0,2025-01-01 00:02:15,Harbord St / Clinton St,7155.0,2025-01-01 00:05:28,Harbord St / Clinton St,4422,Member,ICONIC
1,34637678,464,7508.0,2025-01-01 00:14:07,Berkeley St / Dundas St E - SMART,7042.0,2025-01-01 00:21:51,Berkeley St / Dundas St E - SMART,4618,Member,ICONIC
2,34637663,770,7094.0,2025-01-01 00:07:21,14 Arundel Ave,7777.0,2025-01-01 00:20:11,14 Arundel Ave,7975,Member,EFIT G5
3,34637679,559,7032.0,2025-01-01 00:14:09,Augusta Ave / Dundas St W,7541.0,2025-01-01 00:23:28,Augusta Ave / Dundas St W,3179,Member,ICONIC
4,34637658,116,7017.0,2025-01-01 00:02:13,Widmer St / Adelaide St W - SMART,7102.0,2025-01-01 00:04:09,Widmer St / Adelaide St W - SMART,3177,Member,ICONIC


In [7]:
# Quick data overview
print("Columns:", rides.columns.tolist())
print(f"\nNull counts:\n{rides.isnull().sum()}")
print(f"\nUser types: {rides['User_Type'].value_counts().to_dict()}")
print(f"Unique stations (start): {rides['Start_Station_Id'].nunique()}")
print(f"Dtypes:\n{rides.dtypes}")

Columns: ['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time', 'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name', 'Bike_Id', 'User_Type', 'Bike_Model']

Null counts:
Trip_Id                  0
Trip_Duration            0
Start_Station_Id         7
Start_Time               0
Start_Station_Name       7
End_Station_Id        5844
End_Time               944
End_Station_Name         7
Bike_Id                  0
User_Type                0
Bike_Model               0
dtype: int64

User types: {'Member': 5054784, 'Casual': 2028862}
Unique stations (start): 1031
Dtypes:
Trip_Id                        int64
Trip_Duration                  int64
Start_Station_Id             float64
Start_Time            datetime64[ns]
Start_Station_Name            object
End_Station_Id               float64
End_Time              datetime64[ns]
End_Station_Name              object
Bike_Id                        int64
User_Type                     object
Bike_Model                    obje

## 2. Fetch Station Metadata
Pull station information (capacity, lat/lon) from the Toronto Open Data CKAN API. This gives us station capacity for the capacity gap analysis and coordinates for map visualizations.

In [8]:
# Fetch station metadata from Toronto Open Data CKAN API
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

# Get package metadata for bike-share-toronto
url = base_url + "/api/3/action/package_show"
params = {"id": "bike-share-toronto"}
package = requests.get(url, params=params, timeout=30).json()

# List all resources in the package
for idx, resource in enumerate(package["result"]["resources"]):
    print(f"[{idx}] {resource['name']} — format: {resource.get('format', 'N/A')} — datastore_active: {resource.get('datastore_active', False)}")
    print(f"     id: {resource['id']}")
    print()

[0] bike-share-json — format: JSON — datastore_active: False
     id: 5c1c2c06-d27f-47b7-ae82-926a6d23d76f

[1] bike-share-gbfs-general-bikeshare-feed-specification — format: JSON — datastore_active: False
     id: b69873a1-c180-4ccd-a970-514e434b4971

[2] gbfs-documentation — format: WEB — datastore_active: False
     id: 2eb37596-96a7-4379-b334-a47a396e9ac3

[3] bike-share-stations-readme — format: TXT — datastore_active: False
     id: 5ee32dcb-3f4d-4dfe-a3ad-e27f2c13b5a3



In [15]:
# Fetch the GBFS feed index to find station_information endpoint
gbfs_resource = package["result"]["resources"][1]  # bike-share-gbfs
print(f"Using resource: {gbfs_resource['name']}")

# Get the GBFS index URL and fetch it
gbfs_url = gbfs_resource["url"]
print(f"GBFS index URL: {gbfs_url}")
gbfs_resp = requests.get(gbfs_url, timeout=30)
gbfs_resp.raise_for_status()
gbfs_index = gbfs_resp.json()

# Find the station_information feed URL from the GBFS index
feeds = gbfs_index["data"]["en"]["feeds"]
for feed in feeds:
    print(f"  Feed: {feed['name']} → {feed['url']}")

station_info_url = next(f["url"] for f in feeds if f["name"] == "station_information")
print(f"\nFetching station info from: {station_info_url}")

# Fetch station information
station_resp = requests.get(station_info_url, timeout=30)
station_resp.raise_for_status()
station_json = station_resp.json()

stations = pd.DataFrame(station_json["data"]["stations"])
print(f"\nFetched {len(stations)} stations")
print(f"Columns: {stations.columns.tolist()}")
stations.head()

Using resource: bike-share-gbfs-general-bikeshare-feed-specification
GBFS index URL: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/2b44db0d-eea9-442d-b038-79335368ad5a/resource/b69873a1-c180-4ccd-a970-514e434b4971/download/bike-share-gbfs-general-bikeshare-feed-specification.json
  Feed: system regions → https://tor.publicbikesystem.net/ube/gbfs/v1/en/system_regions
  Feed: system_information → https://tor.publicbikesystem.net/ube/gbfs/v1/en/system_information
  Feed: station_information → https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information
  Feed: station_status → https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_status
  Feed: system_pricing_plans → https://tor.publicbikesystem.net/ube/gbfs/v1/en/system_pricing_plans

Fetching station info from: https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information

Fetched 1016 stations
Columns: ['station_id', 'external_id', 'name', 'physical_configuration', 'lat', 'lon', 'altitude', 'address', 'capacity

,station_id,external_id,name,physical_configuration,lat,lon,altitude,address,capacity,is_charging_station,rental_methods,groups,obcn,short_name,nearby_distance,_ride_code_support,rental_uris,post_code,is_valet_station,cross_street
0,7000,f8aa7900-e9c7-11ef-984e-124c66246c7d,Fort York Blvd / Capreol Ct,REGULAR,43.639832,-79.395954,NaN,Fort York Blvd / Capreol Ct,47,False,"[key, transitcard, creditcard, phone]","[South, Fort York - Entertainment District]",647-643-9607,647-643-9607,500.0,True,{},NaN,NaN,NaN
1,7001,f8aaeb6e-e9c7-11ef-984e-124c66246c7d,Wellesley Station Green P,ELECTRICBIKESTATION,43.664964,-79.383550,NaN,Yonge / Wellesley,23,True,"[key, transitcard, creditcard, phone]","[E-Charging , South, Church Wellesley / Yorkvi...",416-617-9576,416-617-9576,500.0,True,{},M4Y 1G7,NaN,NaN
2,7002,f8aaef03-e9c7-11ef-984e-124c66246c7d,St. George St / Bloor St W,REGULAR,43.667131,-79.399555,NaN,St. George St / Bloor St W,19,False,"[key, transitcard, creditcard, phone]","[South, U of T - Hospital Row]",647-643-9615,647-643-9615,500.0,True,{},NaN,NaN,NaN
3,7003,f8ab20ac-e9c7-11ef-984e-124c66246c7d,Madison Ave / Bloor St W,REGULAR,43.667018,-79.402796,NaN,Madison Ave / Bloor St W,15,False,"[key, transitcard, creditcard, phone]","[South, Bloor St W / Annex]",647-631-4587,647-631-4587,500.0,True,{},NaN,NaN,NaN
4,7005,f8ab51f8-e9c7-11ef-984e-124c66246c7d,King St W / York St,VAULT,43.648001,-79.383177,0.0,King St W / York St,23,False,"[key, transitcard, phone]","[South, Financial District]",647-643-9693,647-643-9693,500.0,True,{},NaN,NaN,NaN


In [16]:
# The GBFS API returns columns: station_id, name, lat, lon, capacity, etc.
# Just keep what we need — no renaming necessary
print(f"Available columns: {stations.columns.tolist()}")

stations["station_id"] = pd.to_numeric(stations["station_id"], errors="coerce").astype("Int64")
stations = stations.dropna(subset=["station_id"])
stations = stations[["station_id", "name", "lat", "lon", "capacity"]].copy()

print(f"\nStations after cleanup: {len(stations)}")
print(f"Capacity range: {stations['capacity'].min()} – {stations['capacity'].max()}")

# Check coverage against ridership data
ride_station_ids = set(rides["Start_Station_Id"].dropna().astype(int).unique())
meta_station_ids = set(stations["station_id"].dropna().unique())
print(f"\nRidership station IDs: {len(ride_station_ids)}")
print(f"Metadata station IDs: {len(meta_station_ids)}")
print(f"Matched: {len(ride_station_ids & meta_station_ids)}")
print(f"In rides but not metadata: {len(ride_station_ids - meta_station_ids)}")
stations.head()

Available columns: ['station_id', 'external_id', 'name', 'physical_configuration', 'lat', 'lon', 'altitude', 'address', 'capacity', 'is_charging_station', 'rental_methods', 'groups', 'obcn', 'short_name', 'nearby_distance', '_ride_code_support', 'rental_uris', 'post_code', 'is_valet_station', 'cross_street']

Stations after cleanup: 1016
Capacity range: 0 – 79

Ridership station IDs: 1031
Metadata station IDs: 1016
Matched: 982
In rides but not metadata: 49


,station_id,name,lat,lon,capacity
0,7000,Fort York Blvd / Capreol Ct,43.639832,-79.395954,47
1,7001,Wellesley Station Green P,43.664964,-79.383550,23
2,7002,St. George St / Bloor St W,43.667131,-79.399555,19
3,7003,Madison Ave / Bloor St W,43.667018,-79.402796,15
4,7005,King St W / York St,43.648001,-79.383177,23


## 3. Fetch Historical Weather Data
Pull hourly weather data for Toronto (Jan 1 – Oct 31, 2025) from the **Open-Meteo Archive API** (free, no key required). We fetch temperature, precipitation, and wind speed.

In [17]:
# Fetch hourly weather data from Open-Meteo Archive API
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

weather_params = {
    "latitude": 43.65,
    "longitude": -79.38,
    "start_date": "2025-01-01",
    "end_date": "2025-10-31",
    "hourly": "temperature_2m,precipitation,wind_speed_10m,weather_code",
    "timezone": "America/Toronto"
}

resp = requests.get(WEATHER_URL, params=weather_params, timeout=60)
resp.raise_for_status()
weather_json = resp.json()

weather = pd.DataFrame({
    "datetime": pd.to_datetime(weather_json["hourly"]["time"]),
    "temperature_c": weather_json["hourly"]["temperature_2m"],
    "precipitation_mm": weather_json["hourly"]["precipitation"],
    "wind_speed_kmh": weather_json["hourly"]["wind_speed_10m"],
    "weather_code": weather_json["hourly"]["weather_code"]
})

print(f"Weather records: {len(weather):,}")
print(f"Date range: {weather['datetime'].min()} → {weather['datetime'].max()}")
print(f"\nNull counts:\n{weather.isnull().sum()}")
print(f"\nTemperature range: {weather['temperature_c'].min():.1f}°C – {weather['temperature_c'].max():.1f}°C")
weather.head()

Weather records: 7,296
Date range: 2025-01-01 00:00:00 → 2025-10-31 23:00:00

Null counts:
datetime            0
temperature_c       0
precipitation_mm    0
wind_speed_kmh      0
weather_code        0
dtype: int64

Temperature range: -20.1°C – 35.8°C


,datetime,temperature_c,precipitation_mm,wind_speed_kmh,weather_code
0,2025-01-01 00:00:00,3.7,0.0,14.9,3
1,2025-01-01 01:00:00,3.3,0.0,14.0,3
2,2025-01-01 02:00:00,2.8,0.0,15.2,3
3,2025-01-01 03:00:00,2.2,0.0,17.9,3
4,2025-01-01 04:00:00,2.0,0.0,19.6,3


## 4. Save Raw Data
Save all three datasets to `data/raw/` for the cleaning notebook to pick up.

In [21]:
# Save raw data
rides.to_csv(RAW_DIR / "rides_raw.csv", index=False)
stations.to_csv(RAW_DIR / "stations.csv", index=False)
weather.to_csv(RAW_DIR / "weather_hourly.csv", index=False)

print("Saved to data/raw/:")
for f in sorted(RAW_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} — {size_mb:.1f} MB")

Saved to data/raw/:
  rides_raw.csv — 999.7 MB
  stations.csv — 0.1 MB
  weather_hourly.csv — 0.3 MB
